In [20]:
import torch
import torch.nn.functional as F
from aeon.datasets import load_from_ts_file
import numpy as np

In [ ]:
# Creates array of tensors
torch.arange(5)

tensor([0, 1, 2, 3, 4])

In [ ]:
# Creates a 1D tensor of 8 steps betweem 0 and 100
indices = torch.linspace(0, 100, 8)
indices

tensor([  0.0000,  14.2857,  28.5714,  42.8571,  57.1429,  71.4286,  85.7143,
        100.0000])

In [ ]:
# Stacks 2 tensors: (i_n, i_{n+1})
intervals_n = torch.stack((indices[:-1], indices[1:]), 1)
intervals_n

tensor([[  0.0000,  14.2857],
        [ 14.2857,  28.5714],
        [ 28.5714,  42.8571],
        [ 42.8571,  57.1429],
        [ 57.1429,  71.4286],
        [ 71.4286,  85.7143],
        [ 85.7143, 100.0000]])

## Testing Loading

In [11]:
base_path = "C:/Users/Gowtham/ROOTDIR/Career/Studies/PG - UCD/Learning/Autumn/AI for Time Series/group project/code/data/Car/Car"

X_train, y_train = load_from_ts_file(f"{base_path}_TRAIN.ts")
X_train.shape, y_train.shape

((60, 1, 577), (60,))

## Make input representations

In [37]:
X = torch.tensor(X_train[0], dtype=torch.float32).unsqueeze(1)

def calculate_input_representations(X):
    original = X
    first_diff = torch.diff(X, dim=-1)
    first_diff_padded = F.pad(first_diff, (2, 2), mode='replicate')
    first_diff_smoothed = F.avg_pool1d(first_diff_padded, kernel_size=5, stride=1)
    second_diff = torch.diff(X, n=2, dim=-1)
    series_fft = torch.fft.rfft(X, dim=-1).abs()
    
    return {
        "original": original,
        "first_diff_smoothed": first_diff_smoothed,
        "second_diff": second_diff,
        "fft": series_fft
    }

input_representations = calculate_input_representations(X)
for key, value in input_representations.items():
    print(f"Shape of {key}: {value.shape}")

Shape of original: torch.Size([1, 1, 577])
Shape of first_diff_smoothed: torch.Size([1, 1, 576])
Shape of second_diff: torch.Size([1, 1, 575])
Shape of fft: torch.Size([1, 1, 289])


## Make intervals

In [39]:
def make_intervals(input_length, depth):
    exponent = min(depth, int(np.log2(input_length)) + 1)
    intervals = []
    
    for dyad_step in 2 ** np.arange(exponent):
        indices = torch.linspace(0, input_length, dyad_step + 1).long()
        dyad_intervals = torch.stack((indices[:-1], indices[1:]), dim=1)
        intervals.append(dyad_intervals)
        
        if dyad_step > 1 and dyad_intervals.diff().median() > 1:
            shift = np.ceil(input_length / (dyad_step * 2))
            shifted_intervals = dyad_intervals[:-1] + shift
            intervals.append(shifted_intervals)
            
    return torch.cat(intervals)
        
intervals = make_intervals(X_train.shape[-1], 3)
intervals

tensor([[  0., 577.],
        [  0., 288.],
        [288., 577.],
        [145., 433.],
        [  0., 144.],
        [144., 288.],
        [288., 432.],
        [432., 577.],
        [ 73., 217.],
        [217., 361.],
        [361., 505.]])

## Extract Quantiles

In [51]:
def extract_quantiles(X_slice, div=4):
    size = X_slice.shape[-1]
    if size == 1:
        return X_slice.flatten(start_dim=1)
    
    num_quantiles = 1 + (size - 1) // div
    if num_quantiles == 1:
        return X_slice.median(dim=1).values.flatten(start_dim=1)
    
    quantile_points = torch.linspace(0, 1, num_quantiles)
    quantiles = torch.quantile(X_slice, quantile_points, dim=-1).permute(1, 2, 0)
    interval_mean = X_slice.mean(dim=-1, keepdim=True)
    quantiles[..., 1::2] -= interval_mean
    
    return quantiles.flatten(start_dim=1)

interval_start, interval_end = intervals[0]
X_slice = X[..., int(interval_start):int(interval_end)]
features = extract_quantiles(X_slice, div=4)

features.shape

torch.Size([1, 145])

# Dataset Upload and Dowload

In [ ]:
# UPLOAD
# from huggingface_hub import upload_folder

# # Push your dataset files
# upload_folder(
#     folder_path="./dataset", repo_id="GowthamNats/NeraThudarDS", repo_type="dataset"
# )

# DOWNLOAD
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="NeraThudar/NeraThudarDS",
#     repo_type="dataset",
#     local_dir="data",
# )
